In [ ]:
import json
import requests

# URL of the VG-SGG dicts file
url = "https://svl.stanford.edu/projects/scene-graph/dataset/VG-SGG-dicts.json"

# Download the file
response = requests.get(url)
response.raise_for_status()  # raise error if download failed

# Parse JSON
vg_dicts = response.json()

# The object categories
object_classes = vg_dicts["idx_to_label"] 
print(list(object_classes.values()))

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import pickle
# Load pre-trained BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

# Example: object labels (single words)
object_labels = list(object_classes.values())

# Tokenize (batch of words)
encoded_inputs = tokenizer(
    object_labels,
    padding=True,          # pad to longest wordpiece sequence
    truncation=True,
    return_tensors="pt"
)

# Forward pass
with torch.no_grad():
    outputs = model(**encoded_inputs)

last_hidden_state = outputs.last_hidden_state  # (batch_size, seq_len, hidden_size)

# To get a single vector per word:
#  - If word is a single token -> just take its hidden state
#  - If word splits into sub-tokens -> average their embeddings
word_embeddings = torch.zeros((151,768))
for i, word in enumerate(object_labels,start=1):
    # get indices of non-padding tokens for this word
    mask = encoded_inputs["attention_mask"][i-1].bool()
    token_embeddings = last_hidden_state[i-1][mask]  # (num_subtokens, hidden_size)
    word_embedding = token_embeddings.mean(dim=0)  # average subtokens
    word_embeddings[i,:] = word_embedding

# Stack into a tensor (num_words, hidden_size)

print("Word embeddings shape:", word_embeddings.shape)  
# e.g. torch.Size([5, 768]) for bert-base-uncased


In [ ]:
save_path = '/kaggle/working/obj_embeddings.pkl'
with open(save_path, "wb") as f:
    pickle.dump(word_embeddings, f)
print(f"✅ Saved embeddings for {word_embeddings.shape} words to {save_path}")